### Imports and Paths

In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

DATA_DIR = "../Datasets/SCANIA"

TRAIN_OPS  = f"{DATA_DIR}/train_operational_readouts.csv"
TRAIN_SPEC = f"{DATA_DIR}/train_specifications.csv"
TRAIN_TTE  = f"{DATA_DIR}/train_tte.csv"

VAL_OPS    = f"{DATA_DIR}/validation_operational_readouts.csv"
VAL_SPEC   = f"{DATA_DIR}/validation_specifications.csv"
VAL_LABELS = f"{DATA_DIR}/validation_labels.csv"

TEST_OPS   = f"{DATA_DIR}/test_operational_readouts.csv"
TEST_SPEC  = f"{DATA_DIR}/test_specifications.csv"
# TEST_LABELS not provided

### Helpers

In [2]:
def last_readout(df_ops: pd.DataFrame) -> pd.DataFrame:
    # keep the last available readout per vehicle (max time_step)
    df_ops = df_ops.sort_values(["vehicle_id", "time_step"])
    return df_ops.drop_duplicates("vehicle_id", keep="last")

def rul_to_ordinal(rul: np.ndarray) -> np.ndarray:
    rul = np.asarray(rul, dtype=float)
    y = np.zeros_like(rul, dtype=int)
    y[rul <= 48] = 1
    y[rul <= 24] = 2
    y[rul <= 12] = 3
    y[rul <=  6] = 4
    return y

# Cost matrix from the screenshot (edit values here if your course uses a different matrix)
COST = np.array([
    [  0,  20,  30,  40,  50],
    [200,   0,  20,  30,  40],
    [300, 200,   0,  20,  30],
    [400, 300, 200,   0,  20],
    [500, 400, 300, 200,   0],
], dtype=int)

def total_cost(y_true, y_pred, cost=COST) -> int:
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    return int(cost[y_true, y_pred].sum())


### Load training data and build ordinal labels

In [3]:
ops_tr  = pd.read_csv(TRAIN_OPS)
spec_tr = pd.read_csv(TRAIN_SPEC)
tte_tr  = pd.read_csv(TRAIN_TTE)

ops_tr_last = last_readout(ops_tr)

df_tr = (
    ops_tr_last
    .merge(spec_tr, on="vehicle_id", how="inner")
    .merge(tte_tr,  on="vehicle_id", how="inner")
)

# Compute RUL at current time t_cur = last readout time_step
t_cur = df_tr["time_step"].to_numpy()
T    = df_tr["length_of_study_time_step"].to_numpy()
rep  = df_tr["in_study_repair"].to_numpy()

# If repaired in-study, use T - t_cur; otherwise right-censored => assign y=0 by default
rul = np.where(rep == 1, np.maximum(T - t_cur, 0.0), np.inf)
df_tr["y"] = np.where(np.isfinite(rul), rul_to_ordinal(rul), 0)

print("Train vehicles:", df_tr.shape[0])
print("Train label counts:\n", df_tr["y"].value_counts().sort_index())
print(df_tr.head())


Train vehicles: 23550
Train label counts:
 y
0    21288
1       31
2       70
3      161
4     2000
Name: count, dtype: int64
   vehicle_id  time_step       171_0     666_0        427_0     837_0  \
0           0      507.4  10189950.0  372685.0          NaN   41670.0   
1           2      281.0   5648790.0  289371.0  233978571.0   68717.0   
2           3      291.2   7603590.0  230831.0  273813365.0  100121.0   
3           4      203.0   4842780.0  210381.0  174662976.0  152385.0   
4           5      357.6   6623040.0  280531.0  246994778.0  164673.0   

     167_0      167_1       167_2       167_3  ...  Spec_1  Spec_2  Spec_3  \
0      NaN        NaN         NaN         NaN  ...    Cat0    Cat0    Cat0   
1  10415.0  9137870.0  74655621.0  45991626.0  ...    Cat1    Cat1    Cat0   
2   5918.0  8225139.0  17004223.0  10504195.0  ...    Cat1    Cat1    Cat1   
3   7128.0  4342398.0  13348382.0  11538870.0  ...    Cat0    Cat2    Cat1   
4      NaN        NaN         NaN         NaN

### Load validation data (last readout only and label)

In [4]:
ops_va  = pd.read_csv(VAL_OPS)
spec_va = pd.read_csv(VAL_SPEC)
lab_va  = pd.read_csv(VAL_LABELS)

ops_va_last = last_readout(ops_va)

# validation_labels.csv should have: vehicle_id, class_label
y_col = "class_label" if "class_label" in lab_va.columns else lab_va.columns[-1]

df_va = (
    ops_va_last
    .merge(spec_va, on="vehicle_id", how="inner")
    .merge(lab_va[["vehicle_id", y_col]], on="vehicle_id", how="inner")
    .rename(columns={y_col: "y"})
)

print("Val vehicles:", df_va.shape[0])
print("Val label counts:\n", df_va["y"].value_counts().sort_index())


Val vehicles: 3027
Val label counts:
 y
0    2944
1      12
2       9
3      21
4      41
Name: count, dtype: int64


### Train a naive Random Forest Model with last readout only

In [5]:
drop_cols = ["vehicle_id", "time_step", "in_study_repair", "length_of_study_time_step"]
X_train = df_tr.drop(columns=drop_cols + ["y"], errors="ignore")
y_train = df_tr["y"].astype(int)

X_val = df_va.drop(columns=drop_cols + ["y"], errors="ignore")
y_val = df_va["y"].astype(int)

cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ],
    remainder="drop",
)

clf = RandomForestClassifier(
    n_estimators=200,
    random_state=0,
    n_jobs=-1,
)

pipe = Pipeline(steps=[("preprocess", preprocess), ("model", clf)])
pipe.fit(X_train, y_train)


,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### Evaluate on validation set

In [6]:
y_pred = pipe.predict(X_val)

tc = total_cost(y_val, y_pred, COST)
print("Total cost:", tc)

print("Accuracy:", accuracy_score(y_val, y_pred))
print("Macro-F1:", f1_score(y_val, y_pred, average="macro"))
print("\nConfusion matrix (rows=true, cols=pred):\n", confusion_matrix(y_val, y_pred))


Total cost: 33370
Accuracy: 0.9666336306574166
Macro-F1: 0.20289169717116584

Confusion matrix (rows=true, cols=pred):
 [[2925    0    0    0   19]
 [  10    0    0    0    2]
 [   9    0    0    0    0]
 [  19    0    0    0    2]
 [  40    0    0    0    1]]


### Compare with dummy model

In [7]:
y_pred_dummy = np.zeros_like(y_val)
tc_dummy = total_cost(y_val, y_pred_dummy, COST)
print("\nDummy total cost (all class 0):", tc_dummy)

print("Accuracy:", accuracy_score(y_val, y_pred_dummy))
print("Macro-F1:", f1_score(y_val, y_pred_dummy, average="macro"))
print("\nConfusion matrix (rows=true, cols=pred):\n", confusion_matrix(y_val, y_pred_dummy))


Dummy total cost (all class 0): 34000
Accuracy: 0.9725801123224315
Macro-F1: 0.19721989616479652

Confusion matrix (rows=true, cols=pred):
 [[2944    0    0    0    0]
 [  12    0    0    0    0]
 [   9    0    0    0    0]
 [  21    0    0    0    0]
 [  41    0    0    0    0]]


### Run on test data, generate csv file to submit

In [8]:
# Load test set
ops_te  = pd.read_csv(TEST_OPS)
spec_te = pd.read_csv(TEST_SPEC)

# Make predictions on test set
ops_te_last = last_readout(ops_te)
df_te = (
    ops_te_last
    .merge(spec_te, on="vehicle_id", how="inner")
)
X_test = df_te.drop(columns=drop_cols, errors="ignore")
y_test_pred = pipe.predict(X_test)

# Save test predictions to CSV, first column is vehicle_id, second column is predicted class label
df_test_pred = pd.DataFrame({
    "id": df_te["vehicle_id"],
    "label": y_test_pred,
})
df_test_pred.to_csv("scania_test_predictions.csv", index=False)


In [9]:
df_test_pred = pd.DataFrame({
    "id": df_te["vehicle_id"],
    "label": np.zeros_like(y_test_pred),
})
df_test_pred.to_csv("scania_test_predictions_dummy.csv", index=False)